# 07 · Perfiles de municipios (v02): logística + hídrica + conectividad

**Objetivo:** generar el perfil de cada municipio y los ficheros `cod_ine → perfil` que usan la API (Adriana) y el RAG (Josema).

**Orden:** después del 05 (y del 06, que registra los experimentos). Lee `modelo_elegido_v02.json`, donde el 05 guarda el K y las entradas elegidos, y los datos de 01A, 03A y 03B. Qué variables entran se explica en el 04; por qué K=4, en el 05.

**Qué cambia respecto a v01:** v01 usaba 7 variables mezcladas + PCA y K=5. v02 usa tres índices 0–100 (logística, hídrica, conectividad), sin PCA, con K=4. Población, paro, viento y riesgos naturales **no** entran en el perfil: quedan como datos sueltos para filtros, scoring y RAG.

El resultado va a `data/reference/` (se sube a Git) con versión `v02`; la API elige sola la versión más alta que tenga los dos ficheros.

> v01 (7 variables + PCA, K=5) sigue en el historial de Git (commit `bc30250`) y en `data/reference/perfiles_*_v01.csv`. Se conserva solo como referencia para el informe.

In [11]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_score

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists())
sys.path.insert(0, str(ROOT / 'src'))  # por si no se ha hecho pip install -e .

from sitewise.ml.indices import INDICES as IDX, construir_indices
from sitewise.ml.perfiles import PERFILES_V02, VERSION_V02, ordenar_clusters_por_indice, tabla_perfiles

DATASET = ROOT / 'data/processed/sitewise_v04_candidata/dataset_municipal.csv'
COMPONENTES = ROOT / 'data/processed/sitewise_dimensiones_v01/componentes_percentiles.csv'
SALIDA = ROOT / 'data/reference'
ML = ROOT / 'data/processed/sitewise_ml'

# Comprobación de orden: el 05 debe haberse ejecutado; de ahí salen K, la semilla y las entradas.
if not (ML / 'modelo_elegido_v02.json').exists():
    raise FileNotFoundError('Falta modelo_elegido_v02.json: ejecuta antes el notebook 05.')
modelo = json.loads((ML / 'modelo_elegido_v02.json').read_text(encoding='utf-8'))
assert modelo['entradas'] == IDX, 'Las entradas guardadas por el 05 no coinciden con los índices del perfil.'
K, SEMILLA = modelo['k'], modelo['semilla']
pd.set_option('display.width', 200)
print('Raíz del proyecto:', ROOT)

Raíz del proyecto: d:\HELEN\PERSONAL\FORMACIÓN\0-ia_formacion\BOOTCAMP - F5\CURSO\PROYECTO_FINAL\SiteWise


## 1. Datos e índices

El cálculo de los tres índices está en `src/sitewise/ml/indices.py` (el mismo que usan los notebooks 05, 06 y 08):

- **Logística** = media de 3 percentiles: suelo industrial (%), distancia a autovía (invertida) y a terminal de mercancías (invertida). **No se imputa**: si falta un componente, el municipio queda sin perfil (Canarias y Baleares no tienen terminal; estimar una distancia por tierra desde un vecino no tendría sentido).
- **Hídrica** = media de 2 percentiles: estrés hídrico y mal estado cuantitativo de acuíferos (más alto = más presión). Los huecos se **estiman con la media de los 5 municipios más cercanos con dato** y se **marcan** como estimados, para que la app no los muestre como dato real.
- **Conectividad** = media de 2 percentiles: fibra FTTH y 5G.

In [12]:
d = pd.read_csv(DATASET, dtype={'cod_ine': str})
componentes = pd.read_csv(COMPONENTES)
assert len(d) == 8132 and d['cod_ine'].is_unique

ind = construir_indices(d, componentes)
d = d.join(ind)
con_perfil = d[IDX].notna().all(axis=1)

print(f'Estimados con vecinos -> acuífero: {int(d.acuifero_estimado_vecinos.sum())} | estrés hídrico: {int(d.estres_hidrico_estimado_vecinos.sum())}')
print('Distancia media a los vecinos (acuífero), km: mediana %.1f, p95 %.1f, máx %.1f' % (
    d.dist_vecinos_acuifero_km.median(), d.dist_vecinos_acuifero_km.quantile(.95), d.dist_vecinos_acuifero_km.max()))
print(f'Municipios con perfil: {int(con_perfil.sum())} de {len(d)} ({con_perfil.mean():.1%})')
print('Sin perfil por falta de logística:', int(d['idx_logistica'].isna().sum()))
print('Sin perfil por falta de hídrica o conectividad:', int((d['idx_logistica'].notna() & ~con_perfil).sum()))

Estimados con vecinos -> acuífero: 722 | estrés hídrico: 24
Distancia media a los vecinos (acuífero), km: mediana 11.8, p95 33.1, máx 52.9
Municipios con perfil: 7971 de 8132 (98.0%)
Sin perfil por falta de logística: 161
Sin perfil por falta de hídrica o conectividad: 0


## 2. K-Means sobre los tres índices (K=4, el elegido en el 05)

Los tres índices ya están en la misma escala (0–100), así que no hace falta escalar ni PCA. Los grupos se **numeran por logística, de mayor a menor**: el 0 es siempre el de mejor logística. Por qué K=4 y no otro valor se justifica en el notebook 05.

In [13]:
X = d.loc[con_perfil, IDX].to_numpy()
km = KMeans(K, n_init=20, random_state=SEMILLA).fit(X)
d.loc[con_perfil, 'cluster'] = ordenar_clusters_por_indice(km.labels_, d.loc[con_perfil, 'idx_logistica'])
d['cluster'] = d['cluster'].astype('Int64')

aris = [adjusted_rand_score(km.labels_, KMeans(K, n_init=20, random_state=s).fit(X).labels_) for s in (1, 2, 3)]
print(f'Silhouette: {silhouette_score(X, km.labels_, sample_size=5000, random_state=0):.3f}  (no comparable con v01: otro espacio de variables)')
print(f'Estabilidad entre semillas (ARI mínimo): {min(aris):.2f}')
centroides = d.loc[con_perfil].groupby('cluster')[IDX].mean().round(0)
print(centroides.to_string())
assert centroides['idx_logistica'].is_monotonic_decreasing

Silhouette: 0.285  (no comparable con v01: otro espacio de variables)
Estabilidad entre semillas (ARI mínimo): 0.99
         idx_logistica  idx_hidrica  idx_conectividad
cluster                                              
0                 75.0         40.0              66.0
1                 58.0         80.0              59.0
2                 40.0         42.0              24.0
3                 33.0         43.0              59.0


## 3. Cómo es cada grupo

Medianas de las variables **originales** (en unidades reales) y medias de los tres índices. Los nombres están en `PERFILES_V02` y son **provisionales** (los cierra Gema). Hay que comprobar que cada nombre describe de verdad lo que se ve en esta tabla.

In [14]:
MEDIANAS = {
    'suelo_industrial_cartografiado_pct': 'mediana_suelo_industrial_cartografiado_pct',
    'dist_recta_eje_autovia_km': 'mediana_dist_recta_eje_autovia_km',
    'dist_recta_ttm_km': 'mediana_dist_recta_ttm_km',
    'estres_hidrico_pct': 'mediana_estres_hidrico_pct',
    'cobertura_ftth_pct': 'mediana_cobertura_ftth_pct',
    'cobertura_5g_pct': 'mediana_cobertura_5g_pct',
}
g = d.loc[con_perfil].groupby('cluster')
resumen = pd.DataFrame({'municipios': g.size()})
for orig in MEDIANAS:
    resumen[orig] = g[orig].median().round(2)
for col in IDX:
    resumen[col] = g[col].mean().round(1)
print(resumen.T.to_string())

cluster                                   0        1        2        3
municipios                          1871.00  1587.00  2470.00  2043.00
suelo_industrial_cartografiado_pct     0.55     0.10     0.00     0.00
dist_recta_eje_autovia_km              3.03     6.71    15.05    19.46
dist_recta_ttm_km                     28.82    53.19    62.34    77.30
estres_hidrico_pct                    41.46   294.75    58.36    58.74
cobertura_ftth_pct                    95.92    94.65    75.44    93.55
cobertura_5g_pct                     100.00   100.00    88.62   100.00
idx_logistica                         75.30    57.60    39.60    33.40
idx_hidrica                           40.30    80.30    42.40    42.80
idx_conectividad                      66.50    58.60    24.40    59.30


## 4. Guardar los ficheros

Tres ficheros en `data/reference/`:

- `perfiles_municipios_v02.csv`: `cod_ine, cluster, cluster_perfil, es_atipico, version_modelo` (solo municipios con perfil).
- `perfiles_definicion_v02.csv`: una fila por perfil, con nombre, descripción, nº de municipios y medianas.
- `perfiles_estado_v02.csv`: **todos** los municipios, con si tienen perfil, por qué no, qué datos se estimaron con vecinos y los tres índices (para mostrar bien el dato estimado y explicar los huecos).

> `es_atipico` es 0 para todos: en v02 **no se evalúa**. En v01 salía de un DBSCAN sobre variables con valores extremos (población); los índices de v02 son percentiles 0–100 y no tienen extremos. El campo se mantiene porque la API y la base de datos lo esperan.

In [15]:
SALIDA.mkdir(parents=True, exist_ok=True)
tabla = tabla_perfiles(d.loc[con_perfil, 'cod_ine'], d.loc[con_perfil, 'cluster'], version=VERSION_V02)
tabla = tabla.sort_values('cod_ine').reset_index(drop=True)
assert tabla['cod_ine'].is_unique and sorted(tabla['cluster'].unique()) == list(range(K))
assert not tabla.isna().any().any()
tabla.to_csv(SALIDA / 'perfiles_municipios_v02.csv', index=False, encoding='utf-8', lineterminator='\n')

definicion = pd.DataFrame({
    'cluster': range(K),
    'cluster_perfil': [PERFILES_V02[i]['nombre'] for i in range(K)],
    'descripcion': [PERFILES_V02[i]['descripcion'] for i in range(K)],
    'n_municipios': g.size().to_numpy(),
})
for orig, nuevo in MEDIANAS.items():
    definicion[nuevo] = g[orig].median().round(2).to_numpy()
for col in IDX:
    definicion['media_' + col] = g[col].mean().round(1).to_numpy()
definicion['version_modelo'] = VERSION_V02
definicion.to_csv(SALIDA / 'perfiles_definicion_v02.csv', index=False, encoding='utf-8', lineterminator='\n')

estado = pd.DataFrame({
    'cod_ine': d['cod_ine'],
    'tiene_perfil': con_perfil.to_numpy(),
    'motivo_sin_perfil': np.where(con_perfil, '', np.where(d['idx_logistica'].isna(), 'sin_dato_logistica(terminal)', 'sin_dato_hidrica')),
    'acuifero_estimado_vecinos': d['acuifero_estimado_vecinos'].to_numpy(),
    'estres_hidrico_estimado_vecinos': d['estres_hidrico_estimado_vecinos'].to_numpy(),
    'idx_logistica': d['idx_logistica'].round(2),
    'idx_hidrica': d['idx_hidrica'].round(2),
    'idx_conectividad': d['idx_conectividad'].round(2),
}).sort_values('cod_ine')
estado.to_csv(SALIDA / 'perfiles_estado_v02.csv', index=False, encoding='utf-8', lineterminator='\n')

print('Guardado en', SALIDA)
print(definicion[['cluster', 'cluster_perfil', 'n_municipios']].to_string(index=False))

Guardado en d:\HELEN\PERSONAL\FORMACIÓN\0-ia_formacion\BOOTCAMP - F5\CURSO\PROYECTO_FINAL\SiteWise\data\reference
 cluster                                  cluster_perfil  n_municipios
       0            Buena logística y buena conectividad          1871
       1 Logística media-buena con mucha presión hídrica          1587
       2                 Logística baja y mal conectados          2470
       3             Logística baja pero bien conectados          2043


## 5. Conclusiones

- Perfil = logística + hídrica + conectividad, K=4. Tres entradas, por debajo del máximo de 3–4 que pidió el revisor técnico.
- Los índices de percentil son casi uniformes, así que K-Means *corta un continuo*: los grupos se justifican por su interpretación, no por un silhouette alto.
- Municipios sin logística (sin terminal) **no tienen perfil**; la API responde `perfil_no_disponible` para ellos.
- Pendiente: nombres definitivos (Gema) y revisar los municipios peninsulares sin distancia a terminal.